# Datamine OUTPUT Process Example

This notebook demonstrates how to configure and run the **`output`** process wrapper in `dmstudio`.

## Process Description

## OUTPUT

Output a database file to a system character format file.

The file includes a special version of the Data Definition in front of it. This is used by the [INPDDF](<inpddf.md>) process to re-read the file back into a database. The format of the file produced is described in the INPDDF documentation. For interfacing with other systems, this Data Definition can be suppressed by setting the @**NODD** parameter to '1'.

The accuracy of the numeric values output is maintained, as far as possible, using a format which changes automatically with the magnitude of the number. It is therefore possible to output numbers spanning the whole range between the lowest and highest system values.

By default, all fields are output; however, the required fields, and the order in which they are output, can be chosen using the optional fields F1 \- Fn. This is helpful if the output file would otherwise be too wide for the system to handle correctly. The maximum permitted width of an output system file is 240 characters, except when exporting to CSV using the faster method described below.

If the database file name is a catalogue file (consisting of a series of file names with a field name of '**FILENAM** \- note the prefixed apostrophe), then first the catalogue file is output from the database file &**IN** , and then each file in turn, which is referenced in the catalogue file, is output. The name of each external system file generated will match that of the database file name.

* **Note** (The following message is displayed when using OUTPUT with a catalogue file input: >>> OPERATING ON A CATALOGUE FILE INPUT <<<):

### Exporting to CSV

A faster export method, described below, is invoked whenever the@CSVparameter is set to '1' and a catalog file isnotspecified.

The following additional parameters can be set:

* @**DPLACE** \- set the number of decimal places used in the output data - for example "@DPLACE = 2" outputs up to 2 decimal places. The default of -1 indicates that the best representation for the magnitude of the data is used.

* @**IMPLICIT** \- if "@IMPLICIT=1", and no required fields have been defined, all table fields are output, including implicit fields. The default is '0'.

When called from a script or macro, more than 25 required output fields can be defined using F1-Fnfields. These must be defined consecutively, and without gaps for example, F1,F2,F4 would not be allowed, and reading would stop afterF2.

If required output fields are defined, and a FIELDLST file is specified, then fields from the FIELDLST file are added to the end of those specified using the F1 \- Fn output fields.

In order to reduce the size of the output file, any trailing zeros (after the decimal point) and trailing spaces after text are automatically trimmed.

The maximum permitted width of 240 characters is not applicable when using this export method.

### Input Files:

* **in** (Table):
  Database file to be output. If IN is a catalogue file, then all files in the catalogue
  are output.
  Required=Yes

* **fieldlst** (Undefined):
  File used to supply selected fields.
  Required=No

### Output Files:

### Fields:

* **fields** (Undefined : Undefined):
  Optional first output field. None specified means "all".
  Default=Undefined
  Required=No

* **fieldnam** (Numeric : FIELDLST):
  Field in FIELDLST to identify selected fields.
  Default=Undefined
  Required=No

### Parameters:

* **csv**:

* **Options** (1: Output in comma separated format (0).):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **nodd**:

* **Options** (1: Suppress output of Data Definition (0).):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **dplace**:
  Exporting to CSV only: specify the maximum number of decimal places to export. Options:
  -1: Use the best representation for the magnitude of the data.; 0: Export 0 decimal
  places; 1: Export 1 decimal place.; 2: Export 2 decimal places; 3: Export 3 decimal
  places; 4: Export 4 decimal places; 5: Export 5 decimal places
  Range=-1,5
  Values=-1,0,1,2,3,4,5
  Default=-1
  Required=No

* **implicit**:
  Exporting to CSV only: if no output fields are specified, allows you to either export
  explicit fields only, or explicit and implicit fields. Options: 0: All explicit fields
  are exported.; 1: All fields are exported, including implicit fields.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('output')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute output
print("Running output...")
dm_cmd.output(
    in_i='t_assays',  # required
    # fieldlst_i='optional',  # optional
    # fields_f=['AU'],  # optional
    # fieldnam_f='optional',  # optional
    # csv_p=0,  # optional
    # nodd_p=0,  # optional
    # dplace_p=-1,  # optional
    # implicit_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("output execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_output_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")